# 📊 Model Dynamic Measurement — Ensemble Average vs Stacking Ensemble
**Mục đích:** Đo toàn diện **hai mô hình** — Ensemble Average (EA) và Stacking Ensemble — bao gồm kích thước disk, RAM, tốc độ inference, cấu trúc nội bộ, metrics, và so sánh chi tiết.

> **Chạy tuần tự từ trên xuống dưới.** Không cần cấu hình thêm gì.

In [1]:
import os, sys, json, time, gc, warnings, pickle, importlib
import tracemalloc
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── Đường dẫn project ──────────────────────────────────────────────────
PROJECT_ROOT = os.path.dirname(os.path.abspath('dynamic_measurement.ipynb'))
sys.path.insert(0, PROJECT_ROOT)

# ── Hai model cần đo ───────────────────────────────────────────────────
MODELS = {
    'Ensemble Average': {
        'artifacts_dir': os.path.join(PROJECT_ROOT, 'Machine_learning_artifacts', 'latest'),
        'module': 'Weather_Forcast_App.Machine_learning_model.interface.predictor_by_ensemble_average',
        'class_name': 'WeatherPredictor',
        'short': 'EA',
    },
    'Stacking Ensemble': {
        'artifacts_dir': os.path.join(PROJECT_ROOT, 'Weather_Forcast_App',
                                       'Machine_learning_artifacts', 'stacking_ensemble', 'latest'),
        'module': 'Weather_Forcast_App.Machine_learning_model.interface.predictor_by_stacking_ensemble',
        'class_name': 'WeatherPredictorStacking',
        'short': 'Stacking',
    },
}

R = {}  # Lưu kết quả đo cho từng model

for name, cfg in MODELS.items():
    print(f'{cfg["short"]:>10} artifacts : {cfg["artifacts_dir"]}')
    assert os.path.isdir(cfg['artifacts_dir']), f'NOT FOUND: {cfg["artifacts_dir"]}'

print(f'\nPROJECT_ROOT : {PROJECT_ROOT}')
print(f'Python       : {sys.version}')

        EA artifacts : /media/voanhnhat/SDD_OUTSIDE6/PROJECT_WEATHER_FORCAST/Machine_learning_artifacts/latest
  Stacking artifacts : /media/voanhnhat/SDD_OUTSIDE6/PROJECT_WEATHER_FORCAST/Weather_Forcast_App/Machine_learning_artifacts/stacking_ensemble/latest

PROJECT_ROOT : /media/voanhnhat/SDD_OUTSIDE6/PROJECT_WEATHER_FORCAST
Python       : 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]


---
## 1 · Kích thước file trên Disk

In [2]:
def human_size(n_bytes):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if n_bytes < 1024:
            return f'{n_bytes:.2f} {unit}'
        n_bytes /= 1024
    return f'{n_bytes:.2f} TB'

for name, cfg in MODELS.items():
    art_dir = cfg['artifacts_dir']
    print(f'\n{"="*60}')
    print(f'  {name} — {art_dir}')
    print(f'{"="*60}')

    rows = []
    total = 0
    for fname in sorted(os.listdir(art_dir)):
        fpath = os.path.join(art_dir, fname)
        if os.path.isfile(fpath):
            sz = os.path.getsize(fpath)
            total += sz
            rows.append({'File': fname, 'Kích thước': human_size(sz)})
    rows.append({'File': '📦 TỔNG CỘNG', 'Kích thước': human_size(total)})
    R.setdefault(name, {})['total_disk_bytes'] = total
    print(pd.DataFrame(rows).to_string(index=False))


  Ensemble Average — /media/voanhnhat/SDD_OUTSIDE6/PROJECT_WEATHER_FORCAST/Machine_learning_artifacts/latest
                  File Kích thước
     Feature_list.json   16.12 KB
          Metrics.json    1.59 KB
             Model.pkl   23.79 MB
       Train_info.json    3.39 KB
Transform_pipeline.pkl   24.54 KB
           📦 TỔNG CỘNG   23.83 MB

  Stacking Ensemble — /media/voanhnhat/SDD_OUTSIDE6/PROJECT_WEATHER_FORCAST/Weather_Forcast_App/Machine_learning_artifacts/stacking_ensemble/latest
                  File Kích thước
     Feature_list.json    3.19 KB
          Metrics.json   39.90 MB
             Model.pkl   46.75 MB
             README.md    3.77 KB
       Train_info.json    6.72 KB
Transform_pipeline.pkl   11.34 KB
           📦 TỔNG CỘNG   86.68 MB


---
## 2 · Load Model — Đo thời gian & RAM

In [4]:
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print('⚠️  psutil chưa cài — bỏ qua đo RSS (pip install psutil)')

for name, cfg in MODELS.items():
    print(f'\n{"="*60}')
    print(f'  Loading: {name}')
    print(f'{"="*60}')

    mod = importlib.import_module(cfg['module'])
    PredictorClass = getattr(mod, cfg['class_name'])

    gc.collect()
    tracemalloc.start()
    ram_before = psutil.Process(os.getpid()).memory_info().rss / 1024**2 if HAS_PSUTIL else 0

    t0 = time.perf_counter()
    predictor = PredictorClass.from_artifacts(cfg['artifacts_dir'])
    load_time = time.perf_counter() - t0

    snapshot = tracemalloc.take_snapshot()
    tracemalloc.stop()
    heap_mb = sum(s.size for s in snapshot.statistics('lineno')) / 1024**2

    ram_delta = 0
    if HAS_PSUTIL:
        ram_after = psutil.Process(os.getpid()).memory_info().rss / 1024**2
        ram_delta = ram_after - ram_before

    R[name].update({
        'predictor': predictor,
        'load_time': load_time,
        'heap_mb': heap_mb,
        'ram_delta': ram_delta,
    })

    print(f'  ✅ Load time      : {load_time:.3f}s')
    print(f'  📦 Python heap    : {heap_mb:.1f} MB')
    if HAS_PSUTIL:
        print(f'  🧠 Delta RSS      : {ram_delta:+.1f} MB')


  Loading: Ensemble Average
  ✅ Load time      : 3.818s
  📦 Python heap    : 72.4 MB
  🧠 Delta RSS      : +381.8 MB

  Loading: Stacking Ensemble
  ✅ Load time      : 1.977s
  📦 Python heap    : 117.5 MB
  🧠 Delta RSS      : +508.8 MB


---
## 3 · Thông tin cấu trúc Model

In [5]:
for name in MODELS:
    predictor = R[name]['predictor']
    info = predictor.get_info()
    R[name]['info'] = info
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    print(json.dumps(info, indent=2, default=str))


  Ensemble Average
{
  "model_type": "WeatherEnsembleModel",
  "target_column": "rain_total",
  "n_features": 150,
  "has_pipeline": true,
  "has_feature_builder": true,
  "forecast_horizon": 0,
  "pipeline_info": {
    "is_fitted": true,
    "fitted_at": "2026-02-25T19:43:45.791265",
    "n_steps": 4,
    "steps": [
      "MissingValueHandler",
      "OutlierHandler",
      "CategoricalEncoder",
      "WeatherScaler"
    ],
    "n_features": 150,
    "config": {
      "missing_strategy": "median",
      "scaler_type": "standard",
      "encoding_type": "label",
      "handle_outliers": true
    }
  },
  "trained_at": "2026-02-25 19:44:06",
  "model_config": {
    "type": "ensemble",
    "params": {
      "base_models": [
        {
          "type": "xgboost",
          "params": {
            "n_estimators": 800,
            "learning_rate": 0.02,
            "max_depth": 16,
            "reg_alpha": 0,
            "reg_lambda": 0
          }
        },
        {
          "type": "r

In [6]:
def obj_size_mb(obj):
    try:
        return len(pickle.dumps(obj, protocol=4)) / 1024**2
    except Exception:
        return float('nan')

for name in MODELS:
    predictor = R[name]['predictor']
    print(f'\n{"="*60}')
    print(f'  Sub-models: {name}')
    print(f'{"="*60}')

    rows = []
    if hasattr(predictor, 'model') and hasattr(getattr(predictor, 'model', None), 'models'):
        # ── Ensemble Average ──
        ensemble = predictor.model
        for i, sub in enumerate(ensemble.models):
            inner = getattr(sub, 'model', None)
            n_est, depth = None, None
            for attr in ('n_estimators', 'iterations', 'num_trees'):
                if hasattr(sub, 'params') and isinstance(sub.params, dict):
                    n_est = n_est or sub.params.get(attr)
                if inner is not None and hasattr(inner, attr):
                    n_est = getattr(inner, attr)
            for attr in ('max_depth', 'depth'):
                if hasattr(sub, 'params') and isinstance(sub.params, dict):
                    depth = depth or sub.params.get(attr)
                if inner is not None and hasattr(inner, attr):
                    depth = getattr(inner, attr)
            sz = obj_size_mb(sub)
            rows.append({
                'Role': 'regressor', 'Name': type(sub).__name__,
                'Inner': type(inner).__name__ if inner else 'N/A',
                'n_est': n_est, 'depth': depth,
                'RAM (MB)': f'{sz:.2f}',
            })

    elif hasattr(predictor, 'stacking'):
        # ── Stacking Ensemble ──
        stacking = predictor.stacking
        for nm, mdl in zip(stacking.cls_model_names, stacking.final_cls_models):
            sz = obj_size_mb(mdl)
            rows.append({'Role': 'base_cls', 'Name': nm, 'Inner': type(mdl).__name__,
                         'n_est': getattr(mdl, 'n_estimators', None),
                         'depth': getattr(mdl, 'max_depth', None),
                         'RAM (MB)': f'{sz:.2f}'})
        for nm, mdl in zip(stacking.reg_model_names, stacking.final_reg_models):
            sz = obj_size_mb(mdl)
            rows.append({'Role': 'base_reg', 'Name': nm, 'Inner': type(mdl).__name__,
                         'n_est': getattr(mdl, 'n_estimators', None),
                         'depth': getattr(mdl, 'max_depth', None),
                         'RAM (MB)': f'{sz:.2f}'})
        for label, mdl in [('meta_cls', stacking.meta_cls), ('meta_reg', stacking.meta_reg)]:
            sz = obj_size_mb(mdl)
            rows.append({'Role': label, 'Name': type(mdl).__name__, 'Inner': '-',
                         'n_est': getattr(mdl, 'n_estimators', None),
                         'depth': getattr(mdl, 'max_depth', None),
                         'RAM (MB)': f'{sz:.2f}'})

    R[name]['sub_models'] = rows
    df_sub = pd.DataFrame(rows)
    print(df_sub.to_string(index=False))
    total_sub = sum(float(r['RAM (MB)']) for r in rows if r['RAM (MB)'] not in ('?', 'nan'))
    print(f'\nTổng RAM sub-models: ~{total_sub:.2f} MB')


  Sub-models: Ensemble Average
     Role                Name                 Inner  n_est  depth RAM (MB)
regressor      WeatherXGBoost          XGBRegressor    800     16     0.12
regressor WeatherRandomForest RandomForestRegressor    800     18    22.64
regressor     WeatherLightGBM         LGBMRegressor    800     16     0.02
regressor     WeatherCatBoost     CatBoostRegressor    200      8     0.83

Tổng RAM sub-models: ~23.61 MB

  Sub-models: Stacking Ensemble
    Role           Name                  Inner  n_est  depth RAM (MB)
base_cls        xgb_cls          XGBClassifier  250.0    4.0     0.39
base_cls         rf_cls RandomForestClassifier  200.0    6.0     1.90
base_cls        cat_cls     CatBoostClassifier    NaN    NaN     0.07
base_cls       lgbm_cls         LGBMClassifier  250.0    4.0     0.44
base_reg        xgb_reg           XGBRegressor  250.0    4.0     0.41
base_reg         rf_reg  RandomForestRegressor  200.0    6.0     1.75
base_reg        cat_reg      CatBoostR

---
## 4 · Đo tốc độ Inference (Latency & Throughput)

In [7]:
for name, cfg in MODELS.items():
    art_dir = cfg['artifacts_dir']
    with open(os.path.join(art_dir, 'Feature_list.json')) as f:
        feat_cfg = json.load(f)

    if isinstance(feat_cfg, list):
        features = feat_cfg
    elif isinstance(feat_cfg, dict):
        features = (feat_cfg.get('all_feature_columns')
                    or feat_cfg.get('features')
                    or feat_cfg.get('feature_names')
                    or list(feat_cfg.keys()))

    R[name]['feature_names'] = features
    print(f'{MODELS[name]["short"]:>10} — {len(features)} features, đầu: {features[:3]}')

        EA — 150 features, đầu: ['rain_avg', 'rain_max', 'location_latitude']
  Stacking — 71 features, đầu: ['location_latitude', 'location_longitude', 'temperature_current']


In [8]:
rng = np.random.default_rng(42)
BATCH_SIZES = [1, 10, 100, 500, 1000]
N_REPEAT = 5

def make_dummy(n_rows, features):
    data = {}
    for col in features:
        cl = col.lower()
        if any(x in cl for x in ('latitude', 'longitude', 'lat', 'lon')):
            data[col] = rng.uniform(8.0, 23.5, n_rows)
        elif any(x in cl for x in ('humidity', 'cloud', 'prob')):
            data[col] = rng.uniform(0, 100, n_rows)
        elif any(x in cl for x in ('temp', 'temperature')):
            data[col] = rng.uniform(15, 40, n_rows)
        elif any(x in cl for x in ('pressure',)):
            data[col] = rng.uniform(990, 1025, n_rows)
        elif any(x in cl for x in ('wind', 'speed')):
            data[col] = rng.uniform(0, 30, n_rows)
        elif any(x in cl for x in ('id', 'station', 'name', 'province', 'district', 'source')):
            data[col] = rng.integers(1, 1000, n_rows).astype(str)
        else:
            data[col] = rng.uniform(0, 1, n_rows)
    return pd.DataFrame(data)

for name in MODELS:
    predictor = R[name]['predictor']
    features = R[name]['feature_names']
    try:
        dummy = make_dummy(10, features)
        _ = predictor.predict(dummy)
        print(f'✅ {MODELS[name]["short"]} warm-up OK')
    except Exception as e:
        print(f'⚠️  {MODELS[name]["short"]} warm-up lỗi: {e}')

Không tìm thấy cột thời gian
Không tìm thấy cột thời gian


✅ EA warm-up OK
✅ Stacking warm-up OK


In [9]:
for name in MODELS:
    predictor = R[name]['predictor']
    features = R[name]['feature_names']

    print(f'\n{"="*60}')
    print(f'  Benchmark: {name}')
    print(f'{"="*60}')

    bench_rows = []
    for bs in BATCH_SIZES:
        dummy = make_dummy(bs, features)
        times = []
        for _ in range(N_REPEAT):
            t0 = time.perf_counter()
            try:
                _ = predictor.predict(dummy)
                ok = True
            except Exception as ex:
                ok = False
                err = str(ex)[:80]
            t1 = time.perf_counter()
            if ok:
                times.append(t1 - t0)

        if times:
            mean_ms = np.mean(times) * 1000
            per_row_us = mean_ms / bs * 1000
            throughput = bs / np.mean(times)
            bench_rows.append({
                'Batch': bs, 'Latency (ms)': f'{mean_ms:.2f}',
                'Per-row (µs)': f'{per_row_us:.1f}',
                'Throughput (rows/s)': f'{throughput:,.0f}',
            })
        else:
            bench_rows.append({'Batch': bs, 'Latency (ms)': 'ERROR',
                               'Per-row (µs)': err, 'Throughput (rows/s)': '-'})

    R[name]['bench'] = bench_rows
    print(pd.DataFrame(bench_rows).to_string(index=False))

Không tìm thấy cột thời gian



  Benchmark: Ensemble Average


Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian


 Batch Latency (ms) Per-row (µs) Throughput (rows/s)
     1       398.75     398747.8                   3
    10       372.84      37283.6                  27
   100       403.53       4035.3                 248
   500       407.36        814.7               1,227
  1000       435.54        435.5               2,296

  Benchmark: Stacking Ensemble


Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian


 Batch Latency (ms) Per-row (µs) Throughput (rows/s)
     1       128.05     128050.9                   8
    10       133.38      13338.3                  75
   100       109.11       1091.1                 917
   500       156.98        314.0               3,185
  1000       122.87        122.9               8,139


---
## 5 · Metrics của từng Model (từ Metrics.json)

In [10]:
KEEP = ['MAE','RMSE','R2','Rain_Detection_Accuracy','Rain_Recall','Rain_Precision','Rain_F1','NonZero_MAE']

for name, cfg in MODELS.items():
    art_dir = cfg['artifacts_dir']
    with open(os.path.join(art_dir, 'Metrics.json')) as f:
        metrics = json.load(f)
    R[name]['metrics'] = metrics

    print(f'\n{"="*60}')
    print(f'  Metrics: {name}')
    print(f'{"="*60}')

    rows = []
    for split in ['train', 'valid', 'test']:
        m = metrics.get(split, {})
        row = {'Split': split.upper()}
        for k in KEEP:
            v = m.get(k)
            row[k] = f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append(row)
    print(pd.DataFrame(rows).to_string(index=False))

    diag = metrics.get('diagnostics', {})
    print(f"\n  Overfit status   : {diag.get('overfit_status')}")
    print(f"  Model quality    : {diag.get('model_quality')}")
    print(f"  Quality detail   : {diag.get('quality_detail')}")
    print(f"  Target zero ratio: {diag.get('target_zero_ratio', 0):.1%}")
    print(f"  Applied log1p    : {metrics.get('applied_log_target')}")


  Metrics: Ensemble Average
Split    MAE   RMSE     R2 Rain_Detection_Accuracy Rain_Recall Rain_Precision Rain_F1 NonZero_MAE
TRAIN 0.5107 2.1233 0.0990                  0.9376        None           None    None      3.0092
VALID 1.1705 2.7168 0.1705                  0.9320        None           None    None      1.9886
 TEST 0.4877 1.7944 0.2307                  0.9199        None           None    None      2.6900

  Overfit status   : underfit
  Model quality    : None
  Quality detail   : None
  Target zero ratio: 85.3%
  Applied log1p    : True

  Metrics: Stacking Ensemble
Split    MAE   RMSE     R2 Rain_Detection_Accuracy Rain_Recall Rain_Precision Rain_F1 NonZero_MAE
TRAIN 0.3217 0.4985 0.6715                  0.8640      0.9458         0.8256  0.8816      0.4062
VALID 0.3851 0.5502 0.5493                  0.7964      0.9163         0.7781  0.8416      0.4420
 TEST 0.5133 0.7231 0.4160                  0.8200      0.9394         0.8004  0.8643      0.6103

  Overfit status   :

---
## 6 · Thông tin Training (từ Train_info.json)

In [11]:
SHOW_KEYS = [
    'generated_at', 'input_file', 'target_column',
    'rows_before', 'rows_after_clean', 'training_time_seconds',
    'train_size', 'valid_size', 'test_size',
    'n_features_selected', 'applied_log_target',
]

for name, cfg in MODELS.items():
    art_dir = cfg['artifacts_dir']
    with open(os.path.join(art_dir, 'Train_info.json')) as f:
        ti = json.load(f)
    R[name]['train_info'] = ti

    print(f'\n{"="*60}')
    print(f'  Train info: {name}')
    print(f'{"="*60}')

    for k in SHOW_KEYS:
        v = ti.get(k, ti.get('config', {}).get(k, 'N/A'))
        if v != 'N/A':
            print(f'  {k:<30}: {v}')

    cfg_section = ti.get('config', {})
    if cfg_section:
        print(f'\n  --- Config ---')
        for k, v in cfg_section.items():
            print(f'    {k}: {v}')


  Train info: Ensemble Average
  target_column                 : rain_total

  Train info: Stacking Ensemble
  target_column                 : rain_total


---
## 7 · Kích thước toàn bộ Model object trong RAM (pickle-serialize)

In [12]:
for name, cfg in MODELS.items():
    predictor = R[name]['predictor']
    art_dir = cfg['artifacts_dir']
    model_pkl_path = os.path.join(art_dir, 'Model.pkl')

    print(f'\n{"="*60}')
    print(f'  Pickle size: {name}')
    print(f'{"="*60}')

    print('  Đang serialize toàn bộ predictor...')
    t0 = time.perf_counter()
    sz_bytes = len(pickle.dumps(predictor, protocol=4))
    t_ser = time.perf_counter() - t0

    print(f'  Kích thước pickle (in-memory) : {sz_bytes/1024**2:.2f} MB')
    print(f'  Thời gian serialize           : {t_ser:.2f}s')

    if os.path.isfile(model_pkl_path):
        disk_sz = os.path.getsize(model_pkl_path)
        print(f'  Kích thước Model.pkl trên disk: {disk_sz/1024**2:.2f} MB')
        print(f'  Tỉ lệ RAM / disk              : {sz_bytes/disk_sz:.2f}x')
    else:
        print(f'  ⚠️  Model.pkl không tồn tại trong artifacts dir')

    R[name]['pickle_bytes'] = sz_bytes


  Pickle size: Ensemble Average
  Đang serialize toàn bộ predictor...
  Kích thước pickle (in-memory) : 23.75 MB
  Thời gian serialize           : 0.05s
  Kích thước Model.pkl trên disk: 23.79 MB
  Tỉ lệ RAM / disk              : 1.00x

  Pickle size: Stacking Ensemble
  Đang serialize toàn bộ predictor...
  Kích thước pickle (in-memory) : 46.75 MB
  Thời gian serialize           : 0.05s
  Kích thước Model.pkl trên disk: 46.75 MB
  Tỉ lệ RAM / disk              : 1.00x


---
## 8 · Đo RAM chi tiết từng sub-model

In [13]:
for name in MODELS:
    predictor = R[name]['predictor']
    print(f'\n{"="*60}')
    print(f'  RAM chi tiết: {name}')
    print(f'{"="*60}')
    print(f'  {"Component":<30} {"Pickle MB":>12} {"% of total":>10}')
    print('  ' + '-' * 55)

    parts = []
    if hasattr(predictor, 'model') and hasattr(getattr(predictor, 'model', None), 'models'):
        # EA: predictor.model.models
        for sub in predictor.model.models:
            try:
                sz = len(pickle.dumps(sub, protocol=4)) / 1024**2
            except Exception:
                sz = float('nan')
            parts.append((type(sub).__name__, sz))

    elif hasattr(predictor, 'stacking'):
        stacking = predictor.stacking
        for nm, mdl in zip(stacking.cls_model_names, stacking.final_cls_models):
            try:
                sz = len(pickle.dumps(mdl, protocol=4)) / 1024**2
            except Exception:
                sz = float('nan')
            parts.append((nm, sz))
        for nm, mdl in zip(stacking.reg_model_names, stacking.final_reg_models):
            try:
                sz = len(pickle.dumps(mdl, protocol=4)) / 1024**2
            except Exception:
                sz = float('nan')
            parts.append((nm, sz))
        for label, mdl in [('meta_cls', stacking.meta_cls), ('meta_reg', stacking.meta_reg)]:
            try:
                sz = len(pickle.dumps(mdl, protocol=4)) / 1024**2
            except Exception:
                sz = float('nan')
            parts.append((label, sz))

    # Pipeline
    if hasattr(predictor, 'pipeline') and predictor.pipeline is not None:
        try:
            pip_sz = len(pickle.dumps(predictor.pipeline, protocol=4)) / 1024**2
        except Exception:
            pip_sz = float('nan')
        parts.append(('Pipeline', pip_sz))

    total_parts = sum(s for _, s in parts if not np.isnan(s))
    for comp_name, sz in parts:
        pct = sz / total_parts * 100 if not np.isnan(sz) and total_parts > 0 else 0
        bar = '█' * int(pct / 5)
        print(f'  {comp_name:<30} {sz:>10.2f}  {pct:>7.1f}%  {bar}')
    print('  ' + '-' * 55)
    print(f'  {"TỔNG":<30} {total_parts:>10.2f} MB')


  RAM chi tiết: Ensemble Average
  Component                         Pickle MB % of total
  -------------------------------------------------------
  WeatherXGBoost                       0.12      0.5%  
  WeatherRandomForest                 22.64     95.8%  ███████████████████
  WeatherLightGBM                      0.02      0.1%  
  WeatherCatBoost                      0.83      3.5%  
  Pipeline                             0.02      0.1%  
  -------------------------------------------------------
  TỔNG                                23.64 MB

  RAM chi tiết: Stacking Ensemble
  Component                         Pickle MB % of total
  -------------------------------------------------------
  xgb_cls                              0.39      7.0%  █
  rf_cls                               1.90     33.9%  ██████
  cat_cls                              0.07      1.3%  
  lgbm_cls                             0.44      7.9%  █
  xgb_reg                              0.41      7.2%  █
  rf_reg

---
## 9 · Top features (SHAP-selected)

In [14]:
for name, cfg in MODELS.items():
    art_dir = cfg['artifacts_dir']
    with open(os.path.join(art_dir, 'Feature_list.json')) as f:
        feat_data = json.load(f)

    print(f'\n{"="*60}')
    print(f'  Features: {name}')
    print(f'{"="*60}')

    if isinstance(feat_data, list):
        features = feat_data
        feat_df = pd.DataFrame({'Feature': features, 'Index': range(len(features))})
    elif isinstance(feat_data, dict):
        all_fc = feat_data.get('all_feature_columns')
        if all_fc:
            feat_df = pd.DataFrame({'Feature': all_fc, 'Index': range(len(all_fc))})
        elif all(isinstance(v, (int, float)) for v in feat_data.values()):
            feat_df = pd.DataFrame(list(feat_data.items()), columns=['Feature', 'Importance'])
            feat_df = feat_df.sort_values('Importance', ascending=False).reset_index(drop=True)
        else:
            feats = feat_data.get('features', feat_data.get('feature_names', []))
            feat_df = pd.DataFrame({'Feature': feats})

    print(f'  Tổng số features: {len(feat_df)}')
    print(feat_df.head(20).to_string(index=False))


  Features: Ensemble Average
  Tổng số features: 150
                        Feature  Index
                       rain_avg      0
                       rain_max      1
              location_latitude      2
         wind_direction_current      3
                 wind_speed_max      4
 temperature_max_pct_change_24h      5
  humidity_max_rolling_mean_24h      6
         temperature_avg_lag_2h      7
            humidity_avg_lag_2h      8
             wind_direction_avg      9
            temperature_current     10
                temperature_max     11
             wind_speed_current     12
             location_longitude     13
          humidity_avg_lag_168h     14
temperature_avg_rolling_mean_3h     15
 temperature_max_rolling_max_3h     16
           humidity_avg_diff_6h     17
   humidity_avg_rolling_mean_3h     18
  humidity_avg_rolling_mean_12h     19

  Features: Stacking Ensemble
  Tổng số features: 71
               Feature  Index
     location_latitude      0
    location_

---
## 10 · So sánh tổng kết hai mô hình

In [15]:
print('=' * 72)
print('  📋  SO SÁNH TỔNG KẾT — ENSEMBLE AVERAGE vs STACKING ENSEMBLE')
print('=' * 72)

cmp_rows = []
for name in MODELS:
    r = R[name]
    info = r.get('info', {})
    met = r.get('metrics', {})
    test_m = met.get('test', {})
    diag = met.get('diagnostics', {})
    bench = r.get('bench', [])
    ti = r.get('train_info', {})
    b1 = bench[0] if bench else {}
    b_last = bench[-1] if bench else {}

    cmp_rows.append({
        'Model': MODELS[name]['short'],
        'Model type': info.get('model_type', '?'),
        'Disk total': human_size(r.get('total_disk_bytes', 0)),
        'Load time (s)': f"{r['load_time']:.3f}",
        'Heap (MB)': f"{r['heap_mb']:.1f}",
        'RSS delta (MB)': f"{r['ram_delta']:+.1f}",
        'Pickle (MB)': f"{r.get('pickle_bytes', 0) / 1024**2:.2f}",
        'Features': len(r.get('feature_names', [])),
        'Train time (s)': ti.get('training_time_seconds',
                                  ti.get('config', {}).get('training_time_seconds', 'N/A')),
        'Dataset rows': ti.get('rows_after_clean',
                                ti.get('config', {}).get('rows_after_clean', 'N/A')),
        'Rain_F1 (test)': f"{test_m.get('Rain_F1', 0):.4f}",
        'Rain_Acc (test)': f"{test_m.get('Rain_Detection_Accuracy', 0):.4f}",
        'R² (test)': f"{test_m.get('R2', 0):.4f}",
        'RMSE (test)': f"{test_m.get('RMSE', 0):.4f}",
        'Overfit': diag.get('overfit_status', '?'),
        'Latency bs=1 (ms)': b1.get('Latency (ms)', '?'),
        'Throughput max (rows/s)': b_last.get('Throughput (rows/s)', '?'),
    })

# Bảng so sánh ngang: cột = EA | Stacking
df_cmp = pd.DataFrame(cmp_rows).set_index('Model').T
print(df_cmp.to_string())
print('\n' + '=' * 72)

  📋  SO SÁNH TỔNG KẾT — ENSEMBLE AVERAGE vs STACKING ENSEMBLE
Model                                      EA           Stacking
Model type               WeatherEnsembleModel  stacking_ensemble
Disk total                           23.83 MB           86.68 MB
Load time (s)                           3.818              1.977
Heap (MB)                                72.4              117.5
RSS delta (MB)                         +381.8             +508.8
Pickle (MB)                             23.75              46.75
Features                                  150                 71
Train time (s)                            N/A                N/A
Dataset rows                              N/A                N/A
Rain_F1 (test)                         0.0000             0.8643
Rain_Acc (test)                        0.9199             0.8200
R² (test)                              0.2307             0.4160
RMSE (test)                            1.7944             0.7231
Overfit                     